In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectFromModel

# Load Dataset

In [2]:
data = pd.read_csv("../docs/fraudTrain.csv")
data.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


## Preprocess data

In [3]:
data.shape

(1296675, 23)

In [4]:
data.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='object')

In [5]:
data['state'].unique()

array(['NC', 'WA', 'ID', 'MT', 'VA', 'PA', 'KS', 'TN', 'IA', 'WV', 'FL',
       'CA', 'NM', 'NJ', 'OK', 'IN', 'MA', 'TX', 'WI', 'MI', 'WY', 'HI',
       'NE', 'OR', 'LA', 'DC', 'KY', 'NY', 'MS', 'UT', 'AL', 'AR', 'MD',
       'GA', 'ME', 'AZ', 'MN', 'OH', 'CO', 'VT', 'MO', 'SC', 'NV', 'IL',
       'NH', 'SD', 'AK', 'ND', 'CT', 'RI', 'DE'], dtype=object)

In [7]:
data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

In [8]:
# Convert trans_date_trans_time to datetime format
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])

In [9]:
# Extract time-based features
data['transaction_hour'] = data['trans_date_trans_time'].dt.hour
data['transaction_day'] = data['trans_date_trans_time'].dt.day
data['transaction_month'] = data['trans_date_trans_time'].dt.month

In [10]:
# Calculate age from dob
data['dob'] = pd.to_datetime(data['dob'], errors='coerce')
data['age'] = (data['trans_date_trans_time'] - data['dob']).dt.days // 365

In [11]:
# Calculate distance between cardholder and merchant locations
data['distance'] = np.sqrt((data['lat'] - data['merch_lat'])**2 + (data['long'] - data['merch_long'])**2)

In [12]:
# Drop redundant columns
data.drop(columns=['trans_date_trans_time', 'dob', 'lat', 'long', 'merch_lat', 'merch_long'], inplace=True)

In [13]:
# Encode categorical variables
categorical_cols = ['merchant', 'category', 'gender', 'job']

In [ ]:
ohe = OneHotEncoder(drop='first')
categorical_data = ohe.fit_transform(data[categorical_cols])
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(categorical_data, columns=ohe_feature_names, index=data.index)

In [ ]:
# Step 3: Handle Data Imbalance
X = data.drop(columns=['is_fraud'])
y = data['is_fraud']

In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
# Step 4: Split Data
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

In [ ]:
# Step 5: Build Pipeline with Random Forest
def create_pipeline():
    numeric_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
    scaler = StandardScaler()
    numeric_transformer = Pipeline(steps=[('scaler', scaler)])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols)
        ])

    rf_model = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')

    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', rf_model)])
    return pipeline

In [ ]:
pipeline = create_pipeline()

In [ ]:
# Step 6: Train the Model
pipeline.fit(X_train, y_train)

In [ ]:
# Step 7: Evaluate the Model
y_pred = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_prob))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Fraud', 'Fraud'], yticklabels=['Not Fraud', 'Fraud'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Feature Importance
rf = pipeline.named_steps['classifier']
feature_importances = rf.feature_importances_
features = X_train.columns
feature_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_df.head(20))
plt.title('Top 20 Feature Importances')
plt.show()

In [ ]:
# Save the model
import joblib
joblib.dump(pipeline, 'fraud_detection_model.pkl')